# Stage 15 - a larger cross-encoder under the Stage 8 recipe

Design: `docs/stage15_large_reranker.md`.

## Setup

In [ ]:
DATA_ROOT = '/content/drive/MyDrive/RAG chunk optimize/artifacts'
REPO_URL = 'https://github.com/sfczaa/rag-chunking-ablation.git'
REPO_REF = 'origin/main'     # branch or full commit SHA
CODE_DIR = '/content/rag-chunking-ablation'
ACCOUNT_LABEL = 'A'          # run-log identifier
CLEAR_STALE_LOCK = False     # requires a stopped lock holder
MODE = 'fresh'               # fresh or resume

from google.colab import drive
drive.mount('/content/drive')

import os, shlex, subprocess, sys
if not os.path.isfile(os.path.join(DATA_ROOT, 'RUN_ROOT_ID.json')):
    raise RuntimeError(f'no RUN_ROOT_ID.json under {DATA_ROOT!r} - the shared folder is not mounted at this path.')


def run(cmd):
    """Stream command output and raise on a nonzero exit."""
    proc = subprocess.Popen(shlex.split(cmd), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='', flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f'command failed: {cmd}')


if not os.path.isdir(os.path.join(CODE_DIR, '.git')):
    run(f'git clone -q {REPO_URL} {CODE_DIR}')
run(f'git -C {CODE_DIR} fetch -q origin')
run(f'git -C {CODE_DIR} checkout -q --detach {REPO_REF}')
run(f'git -C {CODE_DIR} log -1 --format=%H')
os.environ['RAG_DATA_ROOT'] = DATA_ROOT
os.environ['PYTHONUNBUFFERED'] = '1'
sys.path.insert(0, CODE_DIR)
os.chdir(CODE_DIR)
GUARD = f' --account {ACCOUNT_LABEL}' + (' --clear-stale-lock' if CLEAR_STALE_LOCK else '')

## Install dependencies

In [ ]:
run('pip install -q -r requirements.txt')

## Preflight

In [ ]:
run('python -u scripts/38_preflight_stage15.py --gpu')

## Train bge-reranker-large

In [ ]:
run(f'python -u scripts/36_train_large_reranker.py --mode {MODE}' + GUARD)

## Stage 6 bench

In [ ]:
run('python -u scripts/37_eval_large_reranker.py' + GUARD)

## Results

In [ ]:
import json, pathlib
from IPython.display import Markdown, display

latest = pathlib.Path(DATA_ROOT) / 'results' / 'latest'
summary = latest / 'stage15_summary.md'
if summary.exists():
    display(Markdown(summary.read_text(encoding='utf-8')))
    print(json.dumps(json.loads((latest / 'stage15_verdict.json').read_text(encoding='utf-8')),
                     indent=2))
else:
    print('no Stage 15 summary - the evaluation did not finish.')